# 03 — Interpretable development baselines

We compare a trivial class-prior predictor with two transparent models:

1. word-unigram TF-IDF + balanced Logistic Regression;
2. word unigram/bigram TF-IDF + balanced Logistic Regression.

TF-IDF is fitted inside each pipeline on `train` only. The frozen unseen-outlet test is not
read during model selection. Logistic Regression is chosen first because every feature has a
class-specific coefficient and the model exposes probabilities (not yet calibrated).

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data/processed/publication_proxy_headlines.csv"

In [2]:
import joblib
import pandas as pd
from sklearn.dummy import DummyClassifier

from bias_dataset.modeling import (
    attach_split, build_word_logistic, load_modeling_data, prediction_metrics
)

SPLIT_PATH = ROOT / "data/splits/pilot_v1.csv"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports/modeling"

## Load only the development partitions

In [3]:
data = load_modeling_data(DATA_PATH)
manifest = pd.read_csv(SPLIT_PATH)
modeled = attach_split(data, manifest)

train = modeled[modeled["split"].eq("train")].copy()
validation = modeled[modeled["split"].eq("validation")].copy()
print({"train": len(train), "validation": len(validation)})

{'train': 419, 'validation': 118}


In [4]:
assert not train["holdout_outlet"].any()
assert not validation["holdout_outlet"].any()
print("PASS: no holdout-outlet rows entered development.")

PASS: no holdout-outlet rows entered development.


## Establish the class-prior baseline

In [5]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(train[["model_text"]], train["weak_label"])
dummy_prediction = dummy.predict(validation[["model_text"]])
dummy_metrics = prediction_metrics(validation["weak_label"], dummy_prediction)
pd.Series(dummy_metrics, name="class_prior")

accuracy                     0.271186
macro_precision              0.054237
macro_recall                 0.200000
macro_f1                     0.085333
weighted_f1                  0.115706
ordinal_mae                  1.033898
within_one_class_accuracy    0.694915
quadratic_weighted_kappa     0.000000
Name: class_prior, dtype: float64

The dummy model ignores words and always predicts the most common training class. Any useful
text model should improve substantially on its macro-F1.

## Train the unigram model

In [6]:
unigram_model = build_word_logistic(ngram_range=(1, 1), min_df=2)
unigram_model.fit(train["model_text"], train["weak_label"])
unigram_prediction = unigram_model.predict(validation["model_text"])
unigram_metrics = prediction_metrics(validation["weak_label"], unigram_prediction)
pd.Series(unigram_metrics, name="word_unigrams")

accuracy                     0.406780
macro_precision              0.412843
macro_recall                 0.408306
macro_f1                     0.406856
weighted_f1                  0.409067
ordinal_mae                  1.135593
within_one_class_accuracy    0.677966
quadratic_weighted_kappa     0.194740
Name: word_unigrams, dtype: float64

Feature notes: lowercasing and Unicode accent stripping reduce accidental sparsity; stopwords
remain because negation and function words may carry framing; `sublinear_tf` prevents repeated
terms from growing linearly; L2 normalization keeps long headlines from dominating by length.

## Train the unigram + bigram model

In [7]:
bigram_model = build_word_logistic(ngram_range=(1, 2), min_df=2)
bigram_model.fit(train["model_text"], train["weak_label"])
bigram_prediction = bigram_model.predict(validation["model_text"])
bigram_metrics = prediction_metrics(validation["weak_label"], bigram_prediction)
pd.Series(bigram_metrics, name="word_1_2grams")

accuracy                     0.398305
macro_precision              0.404053
macro_recall                 0.399423
macro_f1                     0.399454
weighted_f1                  0.401991
ordinal_mae                  1.194915
within_one_class_accuracy    0.652542
quadratic_weighted_kappa     0.126567
Name: word_1_2grams, dtype: float64

Bigrams allow features such as `white house` or `border policy` to differ from their individual
words. `min_df=2` removes one-off phrases that cannot demonstrate a repeatable association in
this small pilot.

## Compare models on the same temporal validation rows

In [8]:
comparison = pd.DataFrame({
    "class_prior": dummy_metrics,
    "word_unigrams": unigram_metrics,
    "word_1_2grams": bigram_metrics,
}).T.sort_values("macro_f1", ascending=False)
comparison.round(3)

,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,ordinal_mae,within_one_class_accuracy,quadratic_weighted_kappa
word_unigrams,0.407,0.413,0.408,0.407,0.409,1.136,0.678,0.195
word_1_2grams,0.398,0.404,0.399,0.399,0.402,1.195,0.653,0.127
class_prior,0.271,0.054,0.200,0.085,0.116,1.034,0.695,0.000


In [9]:
model_candidates = {
    "word_unigrams": unigram_model,
    "word_1_2grams": bigram_model,
}
selected_name = comparison.loc[list(model_candidates), "macro_f1"].idxmax()
selected_model = model_candidates[selected_name]
print("Selected by validation macro-F1:", selected_name)

Selected by validation macro-F1: word_unigrams


## Save the development artifact and validation predictions

In [10]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(selected_model, MODEL_DIR / "selected_development_lr.joblib")
comparison.to_csv(REPORT_DIR / "development_model_comparison.csv")

In [11]:
selected_prediction = selected_model.predict(validation["model_text"])
validation_output = validation[["record_id", "source_id", "headline", "weak_label"]].copy()
validation_output["prediction"] = selected_prediction
validation_output.to_csv(REPORT_DIR / "validation_predictions.csv", index=False)
validation_output.head()

,record_id,source_id,headline,weak_label,prediction
443,bfae61f0a90f3f5d60bf5f4f77103339e927333b2a8180...,reason,"Today in Supreme Court History: August 24, 1946",Lean Right,Lean Right
486,fdbf7aafdab0f6c2a9d70f6c347b36c8b4f8001339ddab...,reason,Trump Threatens U.S. Relationship With Canada ...,Lean Right,Center
509,e88bc5c99d4477d75be414a22d3b158acfa2b32be3a541...,vox,The Supreme Court just revived Trump’s attempt...,Left,Lean Right
512,675868a2710b42df3355f692933e31d889dd0ce60d1add...,vox,The view from a very angry Canada,Left,Left
518,39bcb55c851309ae1962138e58409b56c974f2046225ea...,the_dispatch,The Military and Politics Don’t Mix,Lean Right,Lean Right


## Development conclusion

The selected model is only a choice between two word-feature configurations. Character
n-grams, LinearSVC and ComplementNB belong in the next model-comparison stage. Notebook 04
asks what this first model learned before Notebook 05 opens the OOD test.